In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP05 - TP Gov't Int
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.abac_2025.fy25_list_of_unit
   2. Cost Center Mapping: vw_cost_center_mapping_bootstrap
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Number of third party relationships or engagements which required 1. the arrangement 
   of government issued approval, license and/or permits, etc. OR 2. the third party 
   to apply or interact with a PO on behalf of TD?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU List + Mapping Bootstrap, strictly 
      filtered by 'jurisdiction' for Canada, Hong Kong, and Barbados.
   2. RISK FLAG LOGIC: Flags engagements as high risk if KPI_Number = '8.2' AND 
      Value = 'YES'.
   3. STRING MAPPING: Maps engagements to AUs by checking if the AU_Name exists 
      inside the text of the 'owning_lob' or 'lob_using' columns.
   4. FINAL OUTPUT: Strict 6-column structure, counting the distinct numerical 
      engagements and converting NULL sums to '0'.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data and strictly filter by target jurisdictions
    SELECT 
        TRIM(CAST(business_ID AS STRING)) AS BusinessID,
        TRIM(business_operational_unit_name) AS AU_Name,
        TRIM(business_segment) AS Business_Segment 
    FROM hive_metastore.abac_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
      AND UPPER(TRIM(jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

Mapped_AUs AS (
    -- 2. Grab every AU that currently has cost centers mapped to it
    SELECT DISTINCT AU_ID 
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
),

All_Base_AUs AS (
    -- 3. Combine filtered Master List and Mapping to create the base table
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        mast.Business_Segment,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

Flagged_Engagements AS (
    -- 4. Filter the TP data strictly based on KPI 8.2 and Value = YES
    SELECT DISTINCT
        EngagementNumber,
        owning_lob,
        lob_using
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
      AND TRIM(KPI_Number) = '8.2' 
      AND UPPER(TRIM(Value)) = 'YES'
),

Engagements_By_AU AS (
    -- 5. Map the flagged engagements to the AUs via string matching
    SELECT 
        a.Base_AU_ID,
        COUNT(DISTINCT f.EngagementNumber) AS Match_Count
    FROM All_Base_AUs a
    INNER JOIN Flagged_Engagements f
        ON UPPER(f.owning_lob) LIKE '%' || UPPER(a.AU_Name) || '%'
        OR UPPER(f.lob_using) LIKE '%' || UPPER(a.AU_Name) || '%'
    GROUP BY a.Base_AU_ID
)

-- 6. Final Template: Strict 6-column output
SELECT 
    a.Base_AU_ID AS BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP05' AS QuestionID,               
    COALESCE(CAST(e.Match_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM All_Base_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.Base_AU_ID = e.Base_AU_ID
ORDER BY a.Base_AU_ID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP05 - TP Gov't Int Drill-Down
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.abac_2025.fy25_list_of_unit
   2. Cost Center Mapping: vw_cost_center_mapping_bootstrap
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   PURPOSE: 
   Provides a row-by-row view of every Third Party Engagement mapped to an AU that 
   triggered a count for TP05. Verifies the KPI_Number is 8.2 and the Value is YES.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(business_ID AS STRING)) AS BusinessID,
        TRIM(business_operational_unit_name) AS AU_Name
    FROM hive_metastore.abac_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
      AND UPPER(TRIM(jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

Mapped_AUs AS (
    SELECT DISTINCT AU_ID 
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
),

All_Base_AUs AS (
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

Flagged_Engagements AS (
    SELECT 
        EngagementNumber,
        ThirdPartyName,
        owning_lob,
        lob_using,
        KPI_Number,
        Value
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
      AND TRIM(KPI_Number) = '8.2' 
      AND UPPER(TRIM(Value)) = 'YES'
)

SELECT DISTINCT
    a.Base_AU_ID AS BusinessID,
    a.AU_Name,
    a.In_ABAC_AU_List,
    f.EngagementNumber,
    f.ThirdPartyName,
    f.owning_lob AS Raw_Col_K_owning_lob,
    f.lob_using AS Raw_Col_L_lob_using,
    f.KPI_Number,
    f.Value
    
FROM All_Base_AUs a

-- Inner join via string matching
INNER JOIN Flagged_Engagements f
    ON UPPER(f.owning_lob) LIKE '%' || UPPER(a.AU_Name) || '%'
    OR UPPER(f.lob_using) LIKE '%' || UPPER(a.AU_Name) || '%'
    
ORDER BY a.Base_AU_ID;